In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [0]:
# Event Hubs configuration
EH_NAMESPACE = "bstockevents"
EH_NAME  = "tradetopic"

EH_CONN_STR = "XXXXXXXXXXXX"


# Kafka Consumer configuration
KAFKA_OPTIONS = {
  "kafka.bootstrap.servers"  : f"{EH_NAMESPACE}.servicebus.windows.net:9093",
  "subscribe"                : EH_NAME,
  "kafka.sasl.mechanism"     : "PLAIN",
  "kafka.security.protocol"  : "SASL_SSL",
  "kafka.sasl.jaas.config"   : f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username=\"$ConnectionString\" password=\"{EH_CONN_STR}\";",
  "kafka.request.timeout.ms" : "60000",
  "kafka.session.timeout.ms" : "45000",
  "kafka.heartbeat.interval.ms": "15000", 
  "maxOffsetsPerTrigger"     : "4500",
  "failOnDataLoss"           : 'true',
  "startingOffsets"          : 'earliest'
}



df = spark.readStream.format("kafka")\
          .options(**KAFKA_OPTIONS)\
          .load()

df_parsed = df.selectExpr(
                "CAST(key AS STRING)",
                "CAST(value AS STRING)",
                "topic",
                "partition",
                "offset"
            )


display(
    df_parsed,
    checkpointLocation="/Volumes/dev_stock/bronze/stock_vol/_checkpoint_kafka3/",
    outputMode="append"
)